# 2. parquet-to-iceberg

Con este tomo el parquet que quedó en el bucket `taxis` (paso anterior) y lo guardo como tabla Iceberg en `my-bucket`, registrada en el catálogo de Nessie.

In [1]:
import dlt
import pyarrow.parquet as pq
import s3fs

Nos conectamos a Minio con s3fs para buscar el parquet que dejó el paso 1 en `taxis/taxis_raw/yellow_tripdata/`.

In [2]:
fs = s3fs.S3FileSystem(
    key="admin",
    secret="password",
    client_kwargs={"endpoint_url": "http://minio:9000"},
)

parquet_path = [p for p in fs.ls("taxis/taxis_raw/yellow_tripdata") if p.endswith(".parquet")][0]
print(parquet_path)

taxis/taxis_raw/yellow_tripdata/1788549118.2246978.eda0651839.parquet


La parte importante es el `table_format="iceberg"` del resource, ahí es donde le decimos a dlt que en vez de un parquet plano arme una tabla Iceberg de una vez.

In [3]:
@dlt.resource(name="yellow_tripdata", table_format="iceberg", write_disposition="replace")
def yellow_tripdata_iceberg():
    table = pq.read_table("s3://" + parquet_path, filesystem=fs)
    yield table

`my-bucket` es donde está el warehouse de Nessie (se ve en el docker-compose). El `dataset_name` que le pongamos (`taxis`) es el namespace que va a quedar creado en el catálogo. La conexión a Nessie (sección `iceberg_catalog`) también está en el secrets, no la escribo acá.

In [4]:
pipeline = dlt.pipeline(
    pipeline_name="parquet_to_iceberg",
    destination=dlt.destinations.filesystem(bucket_url="s3://my-bucket"),
    dataset_name="taxis",
)

Corremos el pipeline.

In [5]:
load_info = pipeline.run(yellow_tripdata_iceberg)
print(load_info)

Pipeline parquet_to_iceberg load step finished in 14.78 seconds
1 load package(s) were loaded to destination filesystem and into dataset taxis
The filesystem destination used s3://my-bucket location to store data
Load package 1788549137.9532044 is LOADED and contains no failed jobs
